In [1]:
from pathlib import Path
from collections import defaultdict
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = "/Users/gupta/Documents/DIS-IND/data/imdb"
CHUNK_SIZE = 200_000


# ============================================================
# ANALYZER
# ============================================================

def analyze_dataset(dataset_path, chunk_size=100_000):
    dataset_path = Path(dataset_path)

    csv_files = sorted(dataset_path.glob("*.csv"))
    tbl_files = sorted(dataset_path.glob("*.tbl"))

    if csv_files and tbl_files:
        raise ValueError(
            "Dataset contains both .csv and .tbl files. "
            "Use only one format per dataset."
        )

    if csv_files:
        files = csv_files
        file_type = "csv"
        # The downloaded Sawfish CENSUS CSV files are semicolon-separated.
        separator = ";"
    elif tbl_files:
        files = tbl_files
        file_type = "tbl"
        separator = "|"
    else:
        raise FileNotFoundError(f"No .csv or .tbl files found in {dataset_path}")

    total_rows = 0
    max_rows_per_table = 0
    total_attributes = 0
    all_attribute_distinct_counts = []
    all_dataset_values = set()
    value_to_attributes = defaultdict(set)
    table_reports = []

    total_size_bytes = sum(file.stat().st_size for file in files)
    total_size_mb = total_size_bytes / (1024 * 1024)

    for file_path in files:
        print(f"Analyzing: {file_path.name}")
        table_rows = 0
        table_size_mb = file_path.stat().st_size / (1024 * 1024)

        if file_type == "csv":
            header = pd.read_csv(file_path, sep=separator, nrows=0)
            columns = list(header.columns)
        else:
            sample = pd.read_csv(
                file_path, sep=separator, header=None, nrows=1, dtype=str
            )
            if len(sample.columns) > 0 and sample.iloc[:, -1].isna().all():
                number_of_columns = len(sample.columns) - 1
            else:
                number_of_columns = len(sample.columns)
            columns = [f"column_{i + 1}" for i in range(number_of_columns)]

        number_of_attributes = len(columns)
        total_attributes += number_of_attributes
        distinct_values = {column: set() for column in columns}

        if file_type == "csv":
            reader = pd.read_csv(
                file_path,
                sep=separator,
                chunksize=chunk_size,
                dtype=str,
                keep_default_na=False,
            )
        else:
            reader = pd.read_csv(
                file_path,
                sep=separator,
                header=None,
                chunksize=chunk_size,
                dtype=str,
                keep_default_na=False,
            )

        for chunk in reader:
            if file_type == "tbl" and len(chunk.columns) > len(columns):
                chunk = chunk.iloc[:, :len(columns)]

            chunk.columns = columns
            table_rows += len(chunk)

            for column in columns:
                unique_values = chunk[column].unique()
                distinct_values[column].update(unique_values)
                all_dataset_values.update(unique_values)
                qualified_attribute = f"{file_path.stem}.{column}"
                for value in unique_values:
                    value_to_attributes[value].add(qualified_attribute)

        total_rows += table_rows
        max_rows_per_table = max(max_rows_per_table, table_rows)
        table_distinct_counts = []

        for column in columns:
            count = len(distinct_values[column])
            table_distinct_counts.append(count)
            all_attribute_distinct_counts.append(count)

        table_max_distinct = max(table_distinct_counts, default=0)
        table_avg_distinct = (
            sum(table_distinct_counts) / len(table_distinct_counts)
            if table_distinct_counts
            else 0
        )
        table_reports.append(
            {
                "table": file_path.name,
                "size_mb": table_size_mb,
                "rows": table_rows,
                "attributes": number_of_attributes,
                "max_distinct": table_max_distinct,
                "average_distinct": table_avg_distinct,
            }
        )

    max_distinct_values_per_attribute = max(
        all_attribute_distinct_counts, default=0
    )
    average_distinct_values_per_attribute = (
        sum(all_attribute_distinct_counts) / len(all_attribute_distinct_counts)
        if all_attribute_distinct_counts
        else 0
    )
    total_distinct_values_dataset = len(all_dataset_values)
    number_of_clusters = len(
        {frozenset(attributes) for attributes in value_to_attributes.values()}
    )

    print()
    print("=" * 80)
    print("DATASET SUMMARY")
    print("=" * 80)
    print(f"Number of tables                       : {len(files):,}")
    print(f"Dataset size (MB)                      : {total_size_mb:,.2f}")
    print(f"Total rows                             : {total_rows:,}")
    print(f"Max # rows per table                   : {max_rows_per_table:,}")
    print(f"Total # attributes                     : {total_attributes:,}")
    print(
        "Max # distinct values per attribute    : "
        f"{max_distinct_values_per_attribute:,}"
    )
    print(
        "Average # distinct values per attribute: "
        f"{average_distinct_values_per_attribute:,.2f}"
    )
    print(
        "Total distinct values in dataset       : "
        f"{total_distinct_values_dataset:,}"
    )
    print(f"Total # clusters                       : {number_of_clusters:,}")

    print()
    print("=" * 80)
    print("TABLE SUMMARY")
    print("=" * 80)
    for table in table_reports:
        print(
            f"{table['table']:<35} "
            f"Rows={table['rows']:<12,} "
            f"Attributes={table['attributes']:<6,} "
            f"Size={table['size_mb']:>10,.2f} MB "
            f"MaxDistinct={table['max_distinct']:>12,} "
            f"AvgDistinct={table['average_distinct']:>12,.2f}"
        )

    return {
        "number_of_tables": len(files),
        "dataset_size_mb": total_size_mb,
        "total_rows": total_rows,
        "max_rows_per_table": max_rows_per_table,
        "total_attributes": total_attributes,
        "max_distinct_values_per_attribute": max_distinct_values_per_attribute,
        "average_distinct_values_per_attribute": average_distinct_values_per_attribute,
        "total_distinct_values_dataset": total_distinct_values_dataset,
        "number_of_clusters": number_of_clusters,
        "tables": table_reports,
    }


report = analyze_dataset(DATASET_PATH, chunk_size=CHUNK_SIZE)


Analyzing: IMDB.csv

DATASET SUMMARY
Number of tables                       : 1
Dataset size (MB)                      : 11,578.77
Total rows                             : 36,244,344
Max # rows per table                   : 36,244,344
Total # attributes                     : 94
Max # distinct values per attribute    : 36,244,344
Average # distinct values per attribute: 1,082,083.46
Total distinct values in dataset       : 47,539,971
Total # clusters                       : 6,171

TABLE SUMMARY
IMDB.csv                            Rows=36,244,344   Attributes=94     Size= 11,578.77 MB MaxDistinct=  36,244,344 AvgDistinct=1,082,083.46


In [1]:
import pandas as pd
from pathlib import Path

project_dir = Path("../")

input_file = project_dir / "data/imdb/imdb/IMDB.csv"
output_dir = project_dir / "data/imdb"
output_dir.mkdir(exist_ok=True)

# Percentages of the total file to generate
percentages = [0.10, 0.20, 0.40, 0.60, 0.80]

# Total file size and total row count
total_size_bytes = input_file.stat().st_size
total_rows = sum(1 for _ in open(input_file, encoding="utf-8", errors="replace")) - 1  # minus header

print(f"Total file size : {total_size_bytes / 1e6:,.2f} MB")
print(f"Total rows      : {total_rows:,}")

# Read the file once (or in chunks if it's large)
df = pd.read_csv(input_file)
print(f"Loaded {len(df):,} rows")

for pct in percentages:
    target_bytes = total_size_bytes * pct
    target_rows = int(len(df) * pct)
    output_file = output_dir / f"{input_file.stem}_{int(pct*100)}pct.csv"
    df.iloc[:target_rows].to_csv(output_file, index=False)
    actual_bytes = output_file.stat().st_size
    print(
        f"Created {output_file.name}: "
        f"{target_rows:,} rows, "
        f"{actual_bytes / 1e6:,.2f} MB "
        f"({actual_bytes / total_size_bytes:.1%} of original)"
    )

Total file size : 12,141.22 MB
Total rows      : 36,244,344
Loaded 36,244,344 rows
Created IMDB_10pct.csv: 3,624,434 rows, 2,577.32 MB (21.2% of original)
Created IMDB_20pct.csv: 7,248,868 rows, 4,470.41 MB (36.8% of original)
Created IMDB_40pct.csv: 14,497,737 rows, 8,172.81 MB (67.3% of original)
Created IMDB_60pct.csv: 21,746,606 rows, 11,766.78 MB (96.9% of original)
Created IMDB_80pct.csv: 28,995,475 rows, 15,322.82 MB (126.2% of original)


In [1]:
import pandas as pd
from pathlib import Path

project_dir = Path("../")

input_file = project_dir / "data/imdb/imdb/IMDB.csv"
output_dir = project_dir / "data/imdb"
output_dir.mkdir(parents=True, exist_ok=True)

# Column percentages instead of fixed counts
target_pcts = [0.10, 0.20, 0.40, 0.60, 0.80]

# Load fixed number of rows
df = pd.read_csv(input_file, sep=";")

total_cols = len(df.columns)
print(f"Loaded {len(df):,} rows and {total_cols} columns")

for pct in target_pcts:
    n = int(total_cols * pct)
    n = max(n, 1)                          # always keep at least 1 column
    n = min(n, total_cols)                 # never exceed total

    subset = df.iloc[:, :n]

    output_file = output_dir / f"{input_file.stem}_{int(pct*100)}pct_cols.csv"
    subset.to_csv(output_file, index=False)

    print(
        f"Created {output_file.name}: "
        f"{len(subset):,} rows × {n} columns "
        f"({pct:.0%} of {total_cols} columns)"
    )

/var/folders/cl/x0sfq21n3m75zvtn65kr_53w0000gn/T/ipykernel_1184/8994069.py:14: DtypeWarning: Columns (0: aka_name.csv - column4, 1: aka_name.csv - column5, 2: aka_name.csv - column6, 3: aka_name.csv - column7, 4: aka_name.csv - column8, 5: comp_cast_type.csv - column2, 6: aka_title.csv - column4, 7: aka_title.csv - column7, 8: aka_title.csv - column12, 9: role_type.csv - column2, 10: info_type.csv - column2, 11: movie_info_idx.csv - column4, 12: char_name.csv - column3, 13: char_name.csv - column5, 14: char_name.csv - column6, 15: char_name.csv - column7, 16: keyword.csv - column3, 17: title.csv - column3, 18: title.csv - column7, 19: title.csv - column11, 20: title.csv - column12, 21: company_name.csv - column3, 22: company_name.csv - column5, 23: company_name.csv - column6, 24: company_name.csv - column7, 25: kind_type.csv - column2, 26: company_type.csv - column2, 27: link_type.csv - column2, 28: name.csv - column3, 29: name.csv - column5, 30: name.csv - column6, 31: name.csv - colu

Loaded 36,244,344 rows and 94 columns
Created IMDB_10pct_cols.csv: 36,244,344 rows × 9 columns (10% of 94 columns)
Created IMDB_20pct_cols.csv: 36,244,344 rows × 18 columns (20% of 94 columns)
Created IMDB_40pct_cols.csv: 36,244,344 rows × 37 columns (40% of 94 columns)
Created IMDB_60pct_cols.csv: 36,244,344 rows × 56 columns (60% of 94 columns)
Created IMDB_80pct_cols.csv: 36,244,344 rows × 75 columns (80% of 94 columns)
